In [1]:
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from shapely.strtree import STRtree

df_lines = pd.read_excel(r"C:\Users\Usuario\Desktop\teste.xls")
print(df_lines)

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
           End X        End Y     Start X      Start Y                Layer
0     48469.3696  156111.0344  48486.1426  156128.2003  EMF_civ_vedacao_R1b
1     48469.3696  156087.0344  48469.3696  156111.0344  EMF_civ_vedacao_R1b
2     48483.3250  156208.6981  48494.1215  156251.3530  EMF_civ_vedacao_R1b
3     48486.1426  156128.2003  48507.0738  156139.9426  EMF_civ_vedacao_R1b
4     48494.1215  156251.3530  48538.3463  156285.7066  EMF_civ_vedacao_R1b
...          ...          ...         ...          ...                  ...
6415  50321.4685  155824.8372  50319.9340  155834.0588  EMF_civ_vedacao_R1b
6416  50322.7754  154982.3403  50293.8060  155024.7650  EMF_civ_vedacao_R1b
6417  50326.3528  155900.2080  50329.3260  155923.6687  EMF_civ_vedacao_R1b
6418  50327.3831  155959.2248  50291.7315  155959.2248  EMF_civ_vedacao_R1b
6419  50329.3260  155923.6687  50327.3831  155959.2248  EMF_civ_vedacao_R1b

[6420 rows x 5

In [4]:
def find_intersections(df):
    """
    Simplified version without complex index handling
    Excludes comparisons between lines with the same Layer
    """
    # Create LineString objects
    geometries = []
    layers = []  # Store the layer information for each geometry

    for _, row in df.iterrows():
        geom = LineString([(row['Start X'], row['Start Y']),
                         (row['End X'], row['End Y'])])
        geometries.append(geom)
        layers.append(row['Layer'])  # Store the layer

    # Build spatial index
    tree = STRtree(geometries)

    intersections = []
    n = len(geometries)

    for i in range(n):
        # Query returns indices of potentially intersecting geometries
        candidate_indices = tree.query(geometries[i])

        for j in candidate_indices:
            # Skip if same layer or same line
            if i < j and layers[i] != layers[j]:
                if geometries[i].intersects(geometries[j]):
                    intersection_point = geometries[i].intersection(geometries[j])

                    if intersection_point.geom_type == 'Point':
                        intersections.append({
                            'line1_layer': layers[i],
                            'line2_layer': layers[j],
                            'x': intersection_point.x,
                            'y': intersection_point.y
                        })

    return pd.DataFrame(intersections)

In [ ]:
def analyze_intersections_with_line_counts(df):
    """
    Find intersections and count how many lines pass through each one
    """

    #Finding all intersections
    intersections_df = find_intersections(df)

    if intersections_df.empty:
        return None

    intersections_df.['point_key'] = intersections_df.apply(lambda row: (round(row['x'], 3), round(row['y'], 3)), axis=1)

    intersections_counts = {}
    line_sets = {}

    for _, row in intersections_df.iterrows():
        point_key = row['point_key']
        line1, line2 = row['line1_id'], row['line2_id']

        if point_key not in intersections_counts:
            intersections_counts[point_key] = 0
            line_sets[point_key] = set()

        line_sets[point_key].add(line1)
        line_sets[point_key].add(line2)
        intersections_counts[point_key] =len(line_sets[point_key])

        intersections_counts['line_count'] = intersections_counts['point_key'].map(intersections_counts)
        intersections_counts['line_ids'] = intersections_df['point_key'].map(lambda key: sorted(line_sets[key])

        intersections_df = intersections_df.drop('point_key', axis=1)

    return intersections_counts
